# Corrected Stage D–E evaluation
Loads `train_all_modifications.py` or `train_mag.py` checkpoints, uses `model.eval()`, the stored FITS PSF, the same mappings, and reports skill over zero.

In [ ]:
from pathlib import Path
import torch, torch.nn.functional as F, matplotlib.pyplot as plt
import data
from differentiable_lensing import DifferentiableLensing
from mapping_io import load_mapping_bundle
from psf import build_psf_kernel, apply_psf
from sisr import SISR
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
CHECKPOINT=Path('outputs_corrected/fixed_sis_corrected/checkpoints/best.pt')
MAPPING_DIR=Path('mappings')
PSF_PATH_OVERRIDE=None
VAL_CLASS='no_sub'; VAL_INDEX=0
ckpt=torch.load(CHECKPOINT,map_location=DEVICE); args=ckpt['args']
model=SISR(args['magnification'],args['n_mag'],args['residual_depth'],in_channels=2,latent_channel_count=args['latent_space_size']).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict']); model.eval()
expected={'lens_model':'SIS','theta_e_arcsec':args['theta_e'],'lr_pixel_scale_arcsec':args['resolution'],'hr_pixel_scale_arcsec':args['target_resolution'],'image_shape':args['image_shape'],'target_shape':args['target_shape']}
files={'backward':'sparse_grid_fracs_euclid_backward_bundle.pt','to_log':'scatter_to_log_128_bundle.pt','forward_from_log':'forward_from_log_128_bundle.pt','from_log':'scatter_from_log_128_bundle.pt'}
M={r:load_mapping_bundle(MAPPING_DIR/f,device=DEVICE,expected={**expected,'mapping_role':r})[0] for r,f in files.items()}
lens=DifferentiableLensing(device=DEVICE,alpha=None,target_resolution=args['target_resolution'],target_shape=args['target_shape']).to(DEVICE)
psf=build_psf_kernel('fits',0.16,args['target_resolution'],path=str(PSF_PATH_OVERRIDE or args['psf_path']),source_pixscale_arcsec=args['psf_source_pixscale_arcsec'],device=DEVICE)

In [ ]:
ds=data.LensingDataset('val/',[VAL_CLASS],VAL_INDEX+1)
lr=ds[VAL_INDEX].unsqueeze(0).float().to(DEVICE)
if lr.ndim==5: lr=lr.squeeze(1)
# Row-normalize the backward map to preserve surface-brightness scale.
b=M['backward'].coalesce(); idx,val=b.indices(),b.values().clamp_min(0); sums=torch.zeros(b.shape[0],device=DEVICE); sums.scatter_add_(0,idx[0],val); bavg=torch.sparse_coo_tensor(idx,val/sums[idx[0]].clamp_min(1e-8),b.shape,device=DEVICE).coalesce()
flat=lr.reshape(lr.shape[0]*lr.shape[1],-1).T
source_lr=torch.sparse.mm(bavg,flat).T.reshape(lr.shape[0],lr.shape[1],args['image_shape'],args['image_shape'])
with torch.no_grad():
    source_hr=model(torch.cat([source_lr,lr],dim=1))
    intrinsic=lens.cross_grid_fill(source_hr,[M['to_log'],M['forward_from_log'],M['from_log']])
    convolved=apply_psf(intrinsic,psf)
    pred=F.interpolate(convolved,size=lr.shape[-2:],mode='area')
zero_mse=lr.square().mean(); model_mse=F.mse_loss(pred,lr); skill=1-model_mse/zero_mse.clamp_min(1e-8)
print({'zero_mse':float(zero_mse),'model_mse':float(model_mse),'skill_over_zero':float(skill)})
fig,ax=plt.subplots(1,3,figsize=(12,4))
for a,x,t in zip(ax,[lr[0,0],pred[0,0],pred[0,0]-lr[0,0]],['LR observation','Re-degraded prediction','Residual']): a.imshow(x.cpu(),cmap='gray'); a.set_title(t)
plt.tight_layout()